# Module 3 — Backend Development with APIs

## Building APIs using FastAPI

Associated Slides:

Slides 17–22

In [ ]:
!pip install fastapi uvicorn

In [ ]:
ctrl + ~

## First FastAPI Application

In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def root():
    return {"message": "Backend is running"}

## Run Backend Server

In [1]:
print("Run this in terminal:")
print("uvicorn main:app --reload")

Run this in terminal:
uvicorn main:app --reload


## Adding Another Endpoint

In [1]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def root():
    return {"message": "Backend is running"}

@app.get("/hello")
def say_hello():
    return {"message": "Hello from FastAPI"}

# Designing Endpoints

In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/items/{item_id}")
def get_item(item_id: int):
    return {"item_id": item_id}

## Query Parameters

In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/search")
def search_item(name: str):
    return {"search": name}

## POST Requests

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Item(BaseModel):
    name: str
    price: float

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Item(BaseModel):
    name: str
    price: float

@app.post("/items")
def create_item(item: Item):
    return {"received": item}

## Testing Using Swagger UI

In [ ]:
print("Open Swagger UI:")
print("http://127.0.0.1:8000/docs")

# Business Logic

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Item(BaseModel):
    name: str
    price: float
    quantity: int

@app.post("/items")
def create_item(item: Item):
    total = item.price * item.quantity

    return {
        "item": item.name,
        "total": total
    }

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Item(BaseModel):
    name: str
    price: float
    quantity: int

@app.post("/items")
def create_item(item: Item):

    total = item.price * item.quantity

    if total > 10000:
        discount = 0.1 * total
        total = total - discount

        return {
            "total": total,
            "discount_applied": True
        }
    
    return {
        "total": total,
        "discount_applied": False
    }

## Separating Logic from Routes

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Item(BaseModel):
    name: str
    price: float
    quantity: int

def calculate_price(price: float, quantity: int):

    total = price * quantity
    discount = 0

    if total > 10000:
        discount = 0.1 * total
        total = total - discount

    return {
        "total": total,
        "discount": discount
    }

@app.post("/items")
def create_item(item: Item):

    pricing = calculate_price(
        item.price,
        item.quantity
    )

    return {
        "item": item.name,
        **pricing
    }

In [ ]:
note:
** in Python is called dictionary unpacking.

It takes all key-value pairs from a dictionary and inserts them into another dictionary.

Example:

pricing = {
    "total": 10800,
    "discount": 1200
}

result = {
    "item": "Laptop",
    **pricing
}

print(result)

## Filtering Logic

In [ ]:
from fastapi import FastAPI

app = FastAPI()

items = [
    {"name": "Laptop", "price": 50000},
    {"name": "Phone", "price": 20000},
    {"name": "Mouse", "price": 500}
]

@app.get("/items")
def get_items(min_price: float = 0):

    results = [
        item for item in items
        if item["price"] >= min_price
    ]

    return {"items": results}

# Error Handling

In [ ]:
from fastapi import FastAPI, HTTPException

app = FastAPI()

items = {
    1: "Laptop",
    2: "Phone"
}

@app.get("/items/{item_id}")
def get_item(item_id: int):

    if item_id not in items:

        raise HTTPException(
            status_code=404,
            detail="Item not found"
        )

    return {
        "item": items[item_id]
    }

## Validation Errors

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Item(BaseModel):
    name: str
    price: float

@app.post("/items")
def create_item(item: Item):
    return item

# Final Project — Inventory API

Associated Slides:

Slides 59–61

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI()

# -----------------------------
# Data Model
# -----------------------------

class Item(BaseModel):
    name: str
    price: float
    quantity: int

# -----------------------------
# In-memory Database
# -----------------------------

items_db = {}
current_id = 1

# -----------------------------
# Business Logic
# -----------------------------

def calculate_price(price, quantity):

    total = price * quantity
    discount = 0

    if total > 10000:
        discount = 0.1 * total

    final = total - discount

    return {
        "total": total,
        "discount": discount,
        "final": final
    }

# -----------------------------
# CREATE
# -----------------------------

@app.post("/items")
def create_item(item: Item):

    global current_id

    pricing = calculate_price(
        item.price,
        item.quantity
    )

    items_db[current_id] = {
        "name": item.name,
        "price": item.price,
        "quantity": item.quantity,
        **pricing
    }

    response = {
        "id": current_id,
        **items_db[current_id]
    }

    current_id += 1

    return response

# -----------------------------
# GET ALL
# -----------------------------

@app.get("/items")
def get_all_items(min_price: float = 0):

    result = []

    for item_id, item in items_db.items():

        if item["price"] >= min_price:

            result.append({
                "id": item_id,
                **item
            })

    return result

# -----------------------------
# GET ONE
# -----------------------------

@app.get("/items/{item_id}")
def get_item(item_id: int):

    if item_id not in items_db:

        raise HTTPException(
            status_code=404,
            detail="Item not found"
        )

    return {
        "id": item_id,
        **items_db[item_id]
    }

# -----------------------------
# UPDATE
# -----------------------------

@app.put("/items/{item_id}")
def update_item(item_id: int, item: Item):

    if item_id not in items_db:

        raise HTTPException(
            status_code=404,
            detail="Item not found"
        )

    pricing = calculate_price(
        item.price,
        item.quantity
    )

    items_db[item_id] = {
        "name": item.name,
        "price": item.price,
        "quantity": item.quantity,
        **pricing
    }

    return {
        "message": "Item updated",
        "id": item_id
    }

# -----------------------------
# DELETE
# -----------------------------

@app.delete("/items/{item_id}")
def delete_item(item_id: int):

    if item_id not in items_db:

        raise HTTPException(
            status_code=404,
            detail="Item not found"
        )

    del items_db[item_id]

    return {
        "message": "Item deleted"
    }

In [56]:
# Example: another Python app calling your API

import requests

data = {
    "name": "Gaming PC",
    "price": 20000,
    "quantity": 5
}

response = requests.post(
    "http://127.0.0.1:8000/items",
    json=data
)

response.json()

{'id': 5,
 'name': 'Gaming PC',
 'price': 20000.0,
 'quantity': 5,
 'total': 100000.0,
 'discount': 10000.0,
 'final': 90000.0}

In [64]:
import requests

response = requests.get("http://127.0.0.1:8000/items")
response


<Response [200]>

In [65]:
response.json()

[{'id': 2,
  'name': 'Gaming Keyboard',
  'price': 4000.0,
  'quantity': 2,
  'total': 8000.0,
  'discount': 0,
  'final': 8000.0},
 {'id': 3,
  'name': 'PS5',
  'price': 10000.0,
  'quantity': 1,
  'total': 10000.0,
  'discount': 0,
  'final': 10000.0},
 {'id': 4,
  'name': 'Laprop',
  'price': 5005.0,
  'quantity': 102,
  'total': 510510.0,
  'discount': 51051.0,
  'final': 459459.0}]

In [62]:

# DELETE request
response = requests.delete("http://127.0.0.1:8000/items/5")
print(response.json())


{'detail': 'Item not found'}


In [63]:
# PUT request
updated = {
    "name": "Gaming Keyboard",
    "price": 4000,
    "quantity": 2
}

response = requests.put(
    "http://127.0.0.1:8000/items/2",
    json=updated
)

print(response.json())

{'message': 'Item updated', 'id': 2}
